# 03 · Conexão segura e a primeira aplicação web

**Pré-requisito:** notebooks 00 a 02.

**Nesta etapa:**
- **Exercício 3:** uma conexão robusta, que explica o que deu errado.
- **Streamlit:** transformar um script Python num site que lê e grava no Atlas.
- **Exercício 4:** publicar a “Secretaria online” na internet.

Para publicar, você vai precisar também de uma conta gratuita no **GitHub** (passo a passo na Parte 3).

In [ ]:
%pip install -q "pymongo[srv]" python-dotenv pillow numpy matplotlib requests streamlit pandas

---
## Parte 1 · Por que a URI não pode ficar no código

A URI contém usuário **e** senha. Se ela aparecer num notebook compartilhado, num print ou num repositório público, qualquer pessoa pode ler, alterar ou apagar o seu banco.

| Onde o código roda | Onde guardar a URI | Como ler no Python |
|---|---|---|
| Notebook no VS Code | arquivo `.env` (+ `.gitignore`) | `load_dotenv()` + `os.environ["MONGO_URI"]` |
| App Streamlit no seu computador | `.streamlit/secrets.toml` | `st.secrets["MONGO_URI"]` |
| App Streamlit publicado | *Secrets* do Streamlit Cloud | `st.secrets["MONGO_URI"]` |

---
## Agora vamos fazer o Exercício 3 · Uma conexão que explica os erros

**Você vai usar:** `MongoClient`, `ping`, `try/except`, os erros `OperationFailure`, `ServerSelectionTimeoutError`, `ConfigurationError`.

1. Escreva uma função `conectar(uri)` que tenta conectar com `ping` e, se falhar, imprime uma mensagem **em português** dizendo o que provavelmente aconteceu (senha errada, rede bloqueada ou URI mal copiada). Se der certo, devolve o `client`.
2. Teste com a URI correta, lida do `.env`.
3. **Provoque um erro de propósito:** chame `conectar` com uma cópia da URI em que você trocou a senha por `senhaerrada`. Confira se a mensagem faz sentido.
4. Use o `client` para recriar `escola.alunos` com os 8 alunos e abra a coleção no Atlas (**Clusters → Browse Collections**). Tire um print.

*Dica:* `uri.replace("SuaSenha", "senhaerrada")` cria a URI errada sem você precisar digitá-la.

In [ ]:
import os
from pymongo import MongoClient
from pymongo.errors import ConfigurationError, OperationFailure, ServerSelectionTimeoutError

def conectar(uri):
    ...
# seu código aqui

In [ ]:
# 3.2 Carregar a URI correta e conectar
# seu código aqui

In [ ]:
# 3.3 Provocar o erro de senha
# seu código aqui

In [ ]:
# 3.4 Recriar escola.alunos
# seu código aqui

---
## Parte 2 · Streamlit: um script vira um site

O Streamlit transforma um arquivo `.py` numa página web, sem HTML nem JavaScript:

| Comando | Para que serve |
|---|---|
| `st.title`, `st.write` | títulos e textos |
| `st.text_input`, `st.number_input`, `st.selectbox` | campos de entrada |
| `st.form` + `st.form_submit_button` | agrupa campos e envia ao clicar |
| `st.dataframe`, `st.metric`, `st.image` | mostrar tabelas, números e imagens |
| `st.secrets` | ler a URI sem escrevê-la no código |
| `@st.cache_resource` | abrir a conexão uma vez e reaproveitar |

> **O segredo do Streamlit:** a cada clique, o script roda **inteiro** de novo, de cima para baixo. Por isso a conexão com o banco fica numa função com `@st.cache_resource`; sem ele, cada clique abriria uma conexão nova com o Atlas.

O app fica numa pasta própria, `secretaria`. A primeira célula cria a pasta e, dentro dela, o arquivo de segredos do Streamlit a partir do seu `.env`. A segunda grava o **esqueleto** do app, que você vai completar no Exercício 4.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
os.makedirs("secretaria/.streamlit", exist_ok=True)
with open("secretaria/.streamlit/secrets.toml", "w", encoding="utf-8") as f:
    f.write(f'MONGO_URI = "{os.environ["MONGO_URI"]}"\n')
print("Criados: secretaria/  e  secretaria/.streamlit/secrets.toml")

In [ ]:
%%writefile secretaria/app.py
import pandas as pd
import streamlit as st
from pymongo import MongoClient

st.set_page_config(page_title="Secretaria online")

@st.cache_resource
def conectar():
    client = MongoClient(st.secrets["MONGO_URI"])
    return client["escola"]["alunos"]

alunos = conectar()
st.title("Secretaria online")

# TODO 1: formulário de cadastro (st.form, st.text_input, st.number_input, st.form_submit_button)

# TODO 2: filtro por curso com a opção "Todos" (st.selectbox, alunos.distinct("curso")) e tabela (st.dataframe)

# TODO 3: indicadores com total de alunos e idade média do filtro (st.columns, st.metric)

In [ ]:
%%writefile secretaria/requirements.txt
streamlit
pymongo[srv]
pandas

---
## Parte 3 · Rodar no seu computador e depois publicar

### Etapa A · Rodar no seu computador
1. No VS Code: **Terminal → New Terminal**.
2. Digite:
   ```
   cd secretaria
   python -m streamlit run app.py
   ```
3. O navegador abre o app em `http://localhost:8501`. A cada vez que você salvar o `app.py`, clique em **Rerun** no app.
4. Para parar: no terminal, **Ctrl+C**.

Se aparecer `No module named streamlit`, rode no mesmo terminal: `python -m pip install streamlit "pymongo[srv]" pandas`.

### Etapa B · Publicar na internet
O app publicado roda num servidor do **Streamlit Community Cloud** (gratuito), que pega o código do **GitHub** e lê a URI dos *Secrets*.

```
secretaria/app.py + requirements.txt ──▶ GitHub ──▶ Streamlit Community Cloud ──▶ https://seu-app.streamlit.app
                                                         └─ Secrets: MONGO_URI ──▶ MongoDB Atlas
```

**B1 · Criar a conta no GitHub**
1. Acesse **https://github.com/signup** e crie a conta (e-mail, senha, nome de usuário).
2. Confirme o e-mail.

**B2 · Criar o repositório e subir os arquivos**
1. No GitHub, clique em **+ → New repository**.
2. **Repository name:** `secretaria-online` · deixe **Public** · clique em **Create repository**.
3. Na página do repositório, clique em **uploading an existing file** (ou **Add file → Upload files**).
4. Arraste **`app.py`** e **`requirements.txt`** da pasta `secretaria`. **Nunca** envie a pasta `.streamlit` nem o `.env`: eles têm a sua senha.
5. Clique em **Commit changes**.

**B3 · Criar o app no Streamlit Community Cloud**
1. Acesse **https://share.streamlit.io** e entre com **Continue with GitHub**; autorize o acesso.
2. Clique em **Create app** → escolha implantar a partir de um repositório do GitHub.
3. **Repository:** `seu-usuario/secretaria-online` · **Branch:** `main` · **Main file path:** `app.py`.
4. Abra **Advanced settings** e, em **Secrets**, cole o conteúdo do seu `secretaria/.streamlit/secrets.toml` (uma linha `MONGO_URI = "..."`).
5. Clique em **Deploy**. Em um ou dois minutos, você recebe um link `https://...streamlit.app`.

**Se der erro:** clique em **Manage app** (canto inferior direito) para ver o log. Os mais comuns são biblioteca faltando no `requirements.txt` e `0.0.0.0/0` não liberado no Atlas (notebook 00, Etapa 4).

**Para alterar o app depois:** edite o `app.py` no GitHub (ícone de lápis → **Commit changes**). O Streamlit atualiza sozinho.

---
## Agora vamos fazer o Exercício 4 · Secretaria online

**Você vai usar:** `st.form`, `st.selectbox`, `alunos.distinct("curso")`, `st.metric`, `st.dataframe`, `st.secrets`, `@st.cache_resource`.

Complete os três `TODO` do app (edite a célula `%%writefile secretaria/app.py` acima e rode de novo, ou edite direto o arquivo `secretaria/app.py` no VS Code):

1. **Cadastro:** formulário com nome, idade e curso que insere um aluno no Atlas.
2. **Filtro:** `selectbox` com “Todos” + os cursos existentes; mostre a tabela filtrada (sem o `_id`).
3. **Indicadores:** total de alunos e idade média do filtro atual.
4. **Teste no seu computador** (Parte 3, Etapa A), **publique** (Etapa B) e **envie o link** ao professor.

*Teste final:* cadastre um aluno pelo site e confira, no Atlas (**Browse Collections**), se ele apareceu em `escola.alunos`.

**Próximo:** `04_Banco_de_Imagens_GridFS.ipynb`